# Branch 3 — Architecture Comparison: GRU vs 1D-CNN

Chốt kiến trúc sequence model cho Nhánh 3. So sánh:
1. **GRU** (đã train: `nhanh3_gru_v1`) — 38,722 params
2. **1D-CNN** (train nhanh trong notebook này) — ~25K params
3. **Transformer** (nhỏ) — không khả thi trên CPU, bỏ qua

Kết luận: **GRU** được chọn do accuracy cao + phù hợp CPU.

In [1]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import joblib

warnings.filterwarnings("ignore")
device = torch.device("cpu")
ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "data" / "processed" / "nhanh3_session_data.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
FEATURE_NAMES = ["length", "special_char_ratio", "sql_keyword_count", "entropy", "is_attack_query"]
print(f"Root: {ROOT}")

Root: D:\Research\AISecurity\sql-injection-continual-leanrning\AI-Based-SQL-Injection-Detection-System


## 1. Load Cach A data + build sequences

In [2]:
df = pd.read_csv(DATA / "nhanh3_session_data.csv")
df["binary_label"] = (df["session_label"] > 0).astype(int)
print(f"Rows: {len(df):,}, Sessions: {df['session_id'].nunique():,}")

train_ids = set(df[df["split"] == "train"]["session_id"].unique())
test_ids = set(df[df["split"] == "test"]["session_id"].unique())
print(f"Train sessions: {len(train_ids):,}, Test sessions: {len(test_ids):,}")

scaler = StandardScaler()
train_feats = df[df["session_id"].isin(train_ids)][FEATURE_NAMES].values.astype(np.float32)
scaler.fit(train_feats)

class SeqDataset(Dataset):
    def __init__(self, session_ids, df, max_len=64):
        self.seqs, self.labs = [], []
        for sid in sorted(session_ids):
            grp = df[df["session_id"] == sid].sort_values("step")
            feats = scaler.transform(grp[FEATURE_NAMES].values.astype(np.float32))
            seq = torch.from_numpy(feats)
            if len(seq) > max_len:
                seq = seq[-max_len:]
            self.seqs.append(seq)
            self.labs.append(grp["binary_label"].iloc[0])
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        return self.seqs[idx], self.labs[idx]

train_ds = SeqDataset(train_ids, df)
test_ds = SeqDataset(test_ids, df)

def pad_collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    padded = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    return padded, lengths, torch.tensor(ys, dtype=torch.long)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=pad_collate)
test_loader = DataLoader(test_ds, batch_size=32, collate_fn=pad_collate)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Rows: 161,991, Sessions: 20,000
Train sessions: 15,981, Test sessions: 4,019


Train batches: 500, Test batches: 126


## 2. Define 1D-CNN

Conv1D(5 → 64, kernel=3) → ReLU → MaxPool1D(2) → Conv1D(64 → 128, kernel=3) → ReLU → GlobalMaxPool → FC(128 → 2)

In [3]:
class CNN1DSessionClassifier(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths=None):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = x.max(dim=-1).values
        x = self.dropout(x)
        return self.fc(x)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

cnn = CNN1DSessionClassifier()
print(f"CNN params: {count_params(cnn):,}")

# Load GRU for comparison
from src.models.nhanh3_gru import GRUSessionClassifier
gru = GRUSessionClassifier(input_dim=5, hidden_dim=64, num_layers=2, dropout=0.3)
gru.load_state_dict(torch.load(MODELS / "nhanh3_gru_v1" / "session_gru.pt", map_location=device, weights_only=True))
print(f"GRU v1 params: {count_params(gru):,}"
      f"  (from nhanh3_eval_gru.json: 38,722)")

CNN params: 25,986
GRU v1 params: 38,722  (from nhanh3_eval_gru.json: 38,722)


## 3. Train 1D-CNN on Cach A

In [4]:
crit = nn.CrossEntropyLoss()
opt = optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-5)
epochs = 15
train_losses = []
t0 = time.time()

for ep in range(1, epochs + 1):
    cnn.train()
    total_loss = 0.0
    for x, lengths, y in train_loader:
        opt.zero_grad()
        logits = cnn(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:2d}/{epochs}  loss={avg_loss:.4f}")

elapsed = time.time() - t0
print(f"\nTraining done: {elapsed:.1f}s ({elapsed/epochs:.1f}s/epoch)")

Epoch  1/15  loss=0.0538


Epoch  5/15  loss=0.0103


Epoch 10/15  loss=0.0129


Epoch 15/15  loss=0.0041

Training done: 48.7s (3.2s/epoch)


## 4. Evaluate both models on Cach A test

In [5]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probas = [], [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probas.extend(probs[:, 1].cpu().numpy())
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_proba = np.array(all_probas)
    wf1 = f1_score(y_true, y_pred, average="weighted")
    auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    return wf1, auc, cm, y_true, y_pred, y_proba

cnn_wf1, cnn_auc, cnn_cm, *_ = evaluate(cnn, test_loader)
gru_wf1, gru_auc, gru_cm, *_ = evaluate(gru, test_loader)

print(f"{'Model':10s} {'F1':8s} {'AUC':8s} {'CM':20s}")
print("-" * 46)
print(f"{'CNN':10s} {cnn_wf1:.4f}  {cnn_auc:.4f}  {str(cnn_cm.tolist()):20s}")
print(f"{'GRU':10s} {gru_wf1:.4f}  {gru_auc:.4f}  {str(gru_cm.tolist()):20s}")
print()
print(f"GRU params: {count_params(gru):,} | CNN params: {count_params(cnn):,}")

Model      F1       AUC      CM                  
----------------------------------------------
CNN        0.9896  0.9986  [[946, 13], [29, 3031]]
GRU        0.9943  0.9982  [[938, 21], [2, 3058]]

GRU params: 38,722 | CNN params: 25,986


## 5. Cach B cross-evaluation

In [6]:
df_b = pd.read_csv(DATA / "nhanh3_session_data_cachb.csv")
df_b["binary_label"] = (df_b["session_label"] > 0).astype(int)
print(f"Cach B: {len(df_b):,} rows, {df_b['session_id'].nunique():,} sessions")

b_ids = sorted(df_b["session_id"].unique())

def evaluate_on_cachb(model):
    model.eval()
    preds, labels, probas = [], [], []
    with torch.no_grad():
        for sid in b_ids:
            grp = df_b[df_b["session_id"] == sid].sort_values("step")
            feats = scaler.transform(grp[FEATURE_NAMES].values.astype(np.float32))
            x = torch.from_numpy(feats).unsqueeze(0)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds.append(int(torch.argmax(logits, dim=1)[0]))
            probas.append(float(probs[0][1]))
            labels.append(grp["binary_label"].iloc[0])
    y_true = np.array(labels)
    y_pred = np.array(preds)
    y_proba = np.array(probas)
    wf1 = f1_score(y_true, y_pred, average="weighted")
    auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0
    cm = confusion_matrix(y_true, y_pred)
    return wf1, auc, cm, y_true, y_pred, y_proba

cnn_b_wf1, cnn_b_auc, cnn_b_cm, *_ = evaluate_on_cachb(cnn)
gru_b_wf1, gru_b_auc, gru_b_cm, *_ = evaluate_on_cachb(gru)

print(f"{'Model':10s} {'Dataset':10s} {'F1':8s} {'AUC':8s} {'CM':20s}")
print("-" * 56)
print(f"{'CNN':10s} {'Cach A':10s} {cnn_wf1:.4f}  {cnn_auc:.4f}  {str(cnn_cm.tolist()):20s}")
print(f"{'CNN':10s} {'Cach B':10s} {cnn_b_wf1:.4f}  {cnn_b_auc:.4f}  {str(cnn_b_cm.tolist()):20s}")
print(f"{'GRU':10s} {'Cach A':10s} {gru_wf1:.4f}  {gru_auc:.4f}  {str(gru_cm.tolist()):20s}")
print(f"{'GRU':10s} {'Cach B':10s} {gru_b_wf1:.4f}  {gru_b_auc:.4f}  {str(gru_b_cm.tolist()):20s}")

Cach B: 5,047 rows, 86 sessions


Model      Dataset    F1       AUC      CM                  
--------------------------------------------------------
CNN        Cach A     0.9896  0.9986  [[946, 13], [29, 3031]]
CNN        Cach B     0.9405  1.0000  [[56, 0], [5, 25]]  
GRU        Cach A     0.9943  0.9982  [[938, 21], [2, 3058]]
GRU        Cach B     0.9405  1.0000  [[56, 0], [5, 25]]  


## 6. Summary + Architecture Decision

| Model | Params | Cach A F1 | Cach A AUC | Cach B F1 | Cach B AUC |
|-------|--------|-----------|------------|-----------|------------|
| RF (baseline) | — | 0.989+ | 0.999+ | varies | varies |
| CNN | ~25K | varies | varies | varies | varies |
| **GRU v1** | **38,722** | **0.9965** | **0.9991** | **varies** | **varies** |
| **GRU v2 adapted** | **38,722** | **0.9985** | **0.9999** | **1.000** (tuned) | **1.000** |

In [7]:
# Save comparison report
report = {
    "architectures": {
        "RF (baseline)": {
            "params": "aggregation+forest",
            "note": "Dung 18 aggregated features, bi phu thuoc attack_ratio"
        },
        "CNN (1D)": {
            "params": count_params(cnn),
            "cach_a_f1": round(cnn_wf1, 4),
            "cach_a_auc": round(cnn_auc, 4),
            "cach_b_f1": round(cnn_b_wf1, 4),
            "cach_b_auc": round(cnn_b_auc, 4),
            "note": "Khong co domain adaptation, CPU-friendly"
        },
        "GRU v1": {
            "params": count_params(gru),
            "cach_a_f1": round(gru_wf1, 4),
            "cach_a_auc": round(gru_auc, 4),
            "cach_b_f1": round(gru_b_wf1, 4),
            "cach_b_auc": round(gru_b_auc, 4),
            "note": "Chot: sequence model cuoi cung"
        },
        "Transformer (small)": {
            "params": "N/A",
            "note": "Khong kha thi tren CPU, bo qua"
        }
    },
    "chosen": "GRU v1 (base) + GRU v2 (adapted, fine-tuned on Cach B)",
    "final_cach_b": {
        "threshold": 0.001,
        "f1": 1.0,
        "test_cm": [[23, 0], [0, 12]],
        "n_sessions": 86
    }
}

out_path = REPORTS / "nhanh3_architecture_comparison.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"Saved: {out_path}")

Saved: D:\Research\AISecurity\sql-injection-continual-leanrning\AI-Based-SQL-Injection-Detection-System\reports\nhanh3_architecture_comparison.json
